# NimbusPay ticket triage

Run on a **free Google Colab T4**. This notebook contains the full code but no training data. Upload candidate_data.zip when prompted. It executes the audit, two baselines, five training runs forming four controlled comparisons, selects on dev, reloads the best PEFT adapter, and generates 400 raw test outputs. Save this notebook **with outputs** after execution.

AI assistance: OpenAI Codex helped implement and verify the code; no AI model supplies labels or predictions. Review all code and measured results before submission.

In [ ]:
%pip -q install transformers==4.51.3 peft==0.15.2 accelerate==1.6.0 bitsandbytes==0.48.2 safetensors==0.5.3 huggingface-hub==0.30.2
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0), "Select Runtime > Change runtime type > T4 GPU"

## Input data

The original 1,445 training rows, 200 dev rows and 400 test inputs must stay private. Only train.jsonl and SCHEMA.md feed the cleaner. Dev is scored after each run; test is read only for final generation.

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile
WORK = Path("/content/nimbuspay")
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
if not (WORK/"data/train.jsonl").exists():
    uploaded = files.upload()  # Select candidate_data.zip
    assert "candidate_data.zip" in uploaded, "Upload the provided candidate_data.zip"
    with zipfile.ZipFile("candidate_data.zip") as archive:
        assert set(archive.namelist()) == {"data/train.jsonl", "data/dev.jsonl", "data/test_inputs.jsonl"}
        archive.extractall(WORK)
print("Data uploaded; it will not be included in the submission archive.")

## Source code

Native Qwen chat-template tokenization; prefix alignment assertions; assistant-only loss with padded tokens masked. The sequence limit is measured on raw training data before any experiment, with no truncation. Inference takes the two original input messages and performs ordinary greedy generation with max_new_tokens=200. The rules-in-prompt option is restricted to the reference baseline on dev.

In [ ]:
%%writefile audit.py
"""Training-only, evidence-based repairs. Never used during inference."""
import argparse
import collections
import copy
import csv
import datetime as dt
from decimal import Decimal
import hashlib
import json
from pathlib import Path
import re

KEYS = ['category', 'priority', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human']
NULL_CATEGORIES = {'account_access', 'kyc', 'other'}
CATEGORIES = NULL_CATEGORIES | {'payment_failed', 'refund', 'fraud', 'offers'}
LEGAL = re.compile(r'\b(?:RBI|ombudsman|consumer court|lawyer|legal (?:notice|action))\b', re.I)
REPEAT = re.compile(r'third time|second complaint|already written \d+ times|contacted support about this before|again and again|teesri baar|pehle bhi complaint|baar baar complaint', re.I)
NUM = r'\d[\d,]*(?:\.\d+)?'
UNIT = r'(?:lakhs?|k|rupees|rupaye)'
MONEY = rf'(?:Rs\.?\s*|INR\s*|₹\s*)?{NUM}(?:\s*{UNIT}\b)?'
MONEY_PATTERNS = [
    re.compile(rf'(?:Rs\.?\s*|INR\s*|₹\s*)(?P<n>{NUM})(?:\s*(?P<u>{UNIT})\b)?', re.I),
    re.compile(rf'(?<![\w.])(?P<n>{NUM})\s*(?P<u>{UNIT})\b', re.I),
    re.compile(rf'(?:amount\s*(?:is|was|:)?|total|rakam)\s*(?P<n>{NUM})(?:\s*(?P<u>{UNIT})\b)?', re.I),
    re.compile(rf'(?<![\w.])(?P<n>{NUM})(?:\s*(?P<u>{UNIT})\b)?\s+(?:in total|was the value|is the amount)', re.I),
]
MONTHS = {name.lower(): i for i, names in enumerate([
    ('Jan', 'January'), ('Feb', 'February'), ('Mar', 'March'), ('Apr', 'April'),
    ('May', 'May'), ('Jun', 'June'), ('Jul', 'July'), ('Aug', 'August'),
    ('Sep', 'September'), ('Oct', 'October'), ('Nov', 'November'), ('Dec', 'December')], 1) for name in names}
MONTH_RE = '|'.join(sorted(MONTHS, key=len, reverse=True))


def load(path):
    return [json.loads(s) for s in Path(path).read_text(encoding='utf-8').splitlines() if s.strip()]


def dump_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def dump_jsonl(path, rows):
    Path(path).write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')


def customer_body(ticket):
    body = ticket.split('\n\n', 1)[-1]
    # A quoted-email boundary starts an earlier thread; it is not current evidence.
    body = re.split(r'(?m)^\s*>', body, maxsplit=1)[0]
    return body


def amount_evidence(body):
    text = re.sub(r'My account balance is .*? right now\.|Account mein abhi .*? pade hain\.', '', body, flags=re.I)
    text = re.sub(rf'FYI my daily limit is {MONEY}\.', '', text, flags=re.I)
    matches = []
    for pattern in MONEY_PATTERNS:
        for m in pattern.finditer(text):
            unit = (m.groupdict().get('u') or '').lower()
            value = Decimal(m['n'].replace(',', ''))
            factor = 100_000 if unit.startswith('lakh') else 1_000 if unit == 'k' else 1
            matches.append((m.start(), m.end(), int(value * factor * 100), m.group()))
    # Prefer longest overlapping match so '1.5 lakh' is not read as 1.5 rupees.
    selected = []
    for m in sorted(matches, key=lambda x: -(x[1] - x[0])):
        if not any(m[0] < s[1] and m[1] > s[0] for s in selected):
            selected.append(m)
    values = {m[2] for m in selected}
    return (next(iter(values)) if len(values) == 1 else None), selected


def date_evidence(ticket, body):
    received = dt.date.fromisoformat(re.search(r'Received:\s*(\d{4}-\d{2}-\d{2})', ticket)[1])
    values = set()
    for m in re.finditer(r'\b(\d+)\s*(?:days? ago|din pehle)\b', body, re.I):
        values.add((received - dt.timedelta(days=int(m[1]))).isoformat())
    for pattern, days in [(r'\b(?:day before yesterday|parso)\b', 2), (r'\b(?:yesterday|kal)\b', 1), (r'\b(?:today|aaj)\b', 0)]:
        # Prevent matching 'yesterday' inside 'day before yesterday'.
        text = re.sub(r'\bday before yesterday\b', '', body, flags=re.I) if days == 1 else body
        if re.search(pattern, text, re.I):
            values.add((received - dt.timedelta(days=days)).isoformat())

    def add(day, month, year=None):
        try:
            yr = int(year) if year else received.year
            if yr < 100:
                yr += 2000
            date = dt.date(yr, int(month), int(day))
            if not year and date > received:
                date = date.replace(year=yr - 1)
            values.add(date.isoformat())
        except ValueError:
            pass

    for m in re.finditer(r'\b(\d{1,2})[-/](\d{1,2})[-/](\d{2}|\d{4})\b', body):
        add(m[1], m[2], m[3])
    for m in re.finditer(rf'\b(\d{{1,2}})(?:st|nd|rd|th)?\s+({MONTH_RE})(?:\s+(\d{{4}}))?\b', body, re.I):
        add(m[1], MONTHS[m[2].lower()], m[3])
    for m in re.finditer(rf'\b({MONTH_RE})\s+(\d{{1,2}})(?:st|nd|rd|th)?(?:,?\s+(\d{{4}}))?\b', body, re.I):
        add(m[2], MONTHS[m[1].lower()], m[3])
    return values


def priority(category, amount, body):
    if category == 'fraud' or (amount is not None and amount >= 5_000_000) or LEGAL.search(body):
        return 'P1'
    if category == 'account_access' or (category in {'refund', 'payment_failed'} and amount is not None and amount >= 500_000):
        return 'P2'
    return 'P3'


def valid_label(obj):
    if not isinstance(obj, dict) or set(obj) != set(KEYS):
        return False
    return (obj['category'] in CATEGORIES and obj['priority'] in {'P1','P2','P3'}
            and (obj['amount_paise'] is None or type(obj['amount_paise']) is int)
            and (obj['txn_date'] is None or isinstance(obj['txn_date'], str) and re.fullmatch(r'\d{4}-\d{2}-\d{2}', obj['txn_date']) is not None)
            and (obj['txn_id'] is None or isinstance(obj['txn_id'], str) and re.fullmatch(r'NP\d{10}', obj['txn_id']) is not None)
            and obj['channel'] in {None, 'upi', 'card', 'netbanking', 'wallet'}
            and obj['language'] in {'en', 'hi-en'} and type(obj['needs_human']) is bool)


def audit(train_path, output_dir):
    path, out = Path(train_path), Path(output_dir)
    if path.name != 'train.jsonl':
        raise ValueError('Cleaner only accepts the provided train.jsonl')
    rows = load(path)
    if any(not r['id'].startswith('tr-') for r in rows):
        raise ValueError('Non-training id detected')
    out.mkdir(parents=True, exist_ok=True)
    cleaned, log, evidence, review = [], [], [], []
    seen, counts = {}, collections.Counter()
    for row in rows:
        r = copy.deepcopy(row)
        ticket = r['messages'][1]['content']
        body = customer_body(ticket)
        if re.fullmatch(r'\s*(?:test(?: test 123)?|asdf asdf)?\s*', body, re.I):
            log.append({'id':r['id'], 'action':'drop', 'reason':'empty_or_placeholder_customer_message; label_has_no_evidence'})
            counts['placeholder_ticket'] += 1
            continue
        signature = tuple(m['content'] for m in r['messages'][:2])
        if signature in seen:
            log.append({'id': r['id'], 'action': 'drop', 'reason': f'exact_duplicate_of:{seen[signature]}'})
            counts['duplicate'] += 1
            continue
        seen[signature] = r['id']
        try:
            original = json.loads(r['messages'][-1]['content'])
        except (ValueError, TypeError):
            log.append({'id': r['id'], 'action': 'drop', 'reason': 'truncated_or_invalid_assistant_json; no invented replacement label'})
            counts['invalid_json'] += 1
            continue
        if not isinstance(original, dict) or original.get('category') not in CATEGORIES:
            log.append({'id': r['id'], 'action': 'drop', 'reason': 'unsupported_label_structure_or_category'})
            counts['unsupported_label'] += 1
            continue
        label, reasons = copy.deepcopy(original), []
        if 'prio' in label:
            label['priority'] = {'low':'P3','medium':'P2','high':'P1'}.get(label.pop('prio'))
            reasons.append('migrate_legacy_prio_to_priority')
        if label.get('txn_date') and re.fullmatch(r'\d{2}/\d{2}/\d{4}', label['txn_date']):
            label['txn_date'] = dt.datetime.strptime(label['txn_date'], '%d/%m/%Y').date().isoformat()
            reasons.append('migrate_legacy_date_to_ISO')
        if label['category'] in NULL_CATEGORIES:
            for k in ['amount_paise','txn_date','txn_id','channel']:
                if label.get(k) is not None:
                    label[k] = None
                    reasons.append(f'null_transaction_field:{k}')
        else:
            amount, candidates = amount_evidence(body)
            if len({m[2] for m in candidates}) == 1 and label.get('amount_paise') != amount:
                old = label.get('amount_paise')
                label['amount_paise'] = amount
                reasons.append(f'amount_conversion:{old}->{amount}; evidence:{candidates[0][3]}')
            elif len({m[2] for m in candidates}) > 1:
                review.append({'id':r['id'], 'issue':'multiple_amount_candidates_retained', 'candidates':candidates})
            dates = date_evidence(ticket, body)
            if len(dates) == 1 and label.get('txn_date') not in dates:
                old = label.get('txn_date')
                label['txn_date'] = next(iter(dates))
                reasons.append(f'transaction_date_evidence:{old}->{label["txn_date"]}')
            if not dates and label.get('txn_date') is not None:
                review.append({'id':r['id'], 'issue':'label_date_not_recognized_in_current_body', 'value':label['txn_date']})
            if label.get('txn_id') is not None:
                norm = re.sub(r'[^A-Z0-9]', '', str(label['txn_id']).upper())
                if norm != label['txn_id'] and re.fullmatch(r'NP\d{10}', norm):
                    label['txn_id'] = norm
                    reasons.append('normalize_transaction_id')
        expected_priority = priority(label['category'], label.get('amount_paise'), body)
        if label.get('priority') != expected_priority:
            reasons.append(f'priority_business_rule:{label.get("priority")}->{expected_priority}')
            label['priority'] = expected_priority
        human = label['priority'] == 'P1' or REPEAT.search(body) is not None
        if type(label.get('needs_human')) is not bool or label.get('needs_human') != human:
            reasons.append(f'human_escalation_rule:{label.get("needs_human")}->{human}')
            label['needs_human'] = human
        if not valid_label(label):
            log.append({'id':r['id'], 'action':'drop', 'reason':'remaining_schema_violation'})
            counts['remaining_schema_violation'] += 1
            continue
        if reasons:
            log.append({'id':r['id'], 'action':'fix', 'reason':'; '.join(reasons)})
            evidence.append({'id':r['id'], 'before':original, 'after':label, 'reasons':reasons})
            for reason in reasons:
                counts[reason.split(':')[0]] += 1
        r['messages'][-1]['content'] = json.dumps({k:label[k] for k in KEYS}, ensure_ascii=False)
        cleaned.append(r)
    dump_jsonl(out / 'train_clean.jsonl', cleaned)
    with (out / 'cleaning_log.csv').open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id','action','reason'])
        writer.writeheader()
        writer.writerows(log)
    dump_json(out / 'cleaning_evidence.json', evidence)
    dump_json(out / 'audit_review.json', review)
    summary = {'input_rows':len(rows), 'retained_rows':len(cleaned),
               'dropped_rows':sum(r['action']=='drop' for r in log),
               'fixed_rows':sum(r['action']=='fix' for r in log),
               'findings':dict(counts), 'review_rows':len(review),
               'quoted_thread_rows':sum(bool(re.search(r'(?m)^\s*>', r['messages'][1]['content'])) for r in rows),
               'source_counts':dict(collections.Counter(re.search(r'Source:\s*(\S+)', r['messages'][1]['content'])[1] for r in rows)),
               'category_counts':dict(collections.Counter(json.loads(r['messages'][-1]['content'])['category'] for r in cleaned)),
               'input_sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
    dump_json(out / 'audit_summary.json', summary)
    return summary


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--train', default='data/train.jsonl')
    parser.add_argument('--out', default='artifacts')
    args = parser.parse_args()
    print(json.dumps(audit(args.train, args.out), indent=2))


In [ ]:
%%writefile pipeline.py
"""Colab T4 training and inference; inference contains no labelling rules."""
import argparse
from dataclasses import asdict, dataclass
import gc
import hashlib
import json
import math
from pathlib import Path
import random
import shutil
import time

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
SEED = 2026


@dataclass
class Run:
    name: str
    data: str = 'clean'
    quantized: bool = True
    rank: int = 16
    lr: float = 2e-4
    epochs: int = 2
    assistant_only: bool = True


RUNS = [Run('clean_qlora'), Run('raw_qlora', data='raw'),
        Run('clean_lora', quantized=False), Run('rank8', rank=8),
        Run('full_loss', assistant_only=False)]
COMPARISONS = [('raw_qlora', 'clean_qlora', 'raw vs cleaned training labels'),
               ('clean_lora', 'clean_qlora', '16-bit LoRA vs 4-bit QLoRA'),
               ('rank8', 'clean_qlora', 'rank 8 vs rank 16 (alpha/r held at 2)'),
               ('full_loss', 'clean_qlora', 'all-token vs assistant-only loss')]


def load_rows(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(s) for s in f if s.strip()]


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def free_gpu():
    import torch
    gc.collect()
    torch.cuda.empty_cache()


def load_tokenizer(revision):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision)
    tok.pad_token = tok.eos_token
    return tok


def load_model(quantized, revision):
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    kwargs = dict(revision=revision, torch_dtype=torch.float16,
                  device_map={'':0}, attn_implementation='sdpa')
    if quantized:
        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True,
            bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)


def encode_training(row, tokenizer, assistant_only=True):
    if not row['id'].startswith('tr-'):
        raise ValueError('Training input must contain only training IDs')
    messages = row['messages']
    if [m['role'] for m in messages] != ['system','user','assistant']:
        raise ValueError('Unexpected conversation structure')
    prefix = tokenizer.apply_chat_template(messages[:2], tokenize=True, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
    if full[:len(prefix)] != prefix:
        raise ValueError('Chat template is not prefix aligned; cannot mask safely')
    labels = ([-100]*len(prefix) + full[len(prefix):]) if assistant_only else full.copy()
    assert any(x != -100 for x in labels)
    return {'input_ids':full, 'attention_mask':[1]*len(full), 'labels':labels}


def length_audit(rows, tokenizer, out):
    lengths = sorted(len(encode_training(r, tokenizer)['input_ids']) for r in rows)
    def q(p):
        return lengths[math.ceil(p*len(lengths))-1]
    maximum = lengths[-1]
    selected = math.ceil(maximum/128)*128
    result = {'n':len(lengths), 'p50':q(.5), 'p95':q(.95), 'p99':q(.99),
              'max':maximum, 'sequence_length':selected, 'truncated_training_rows':0}
    write_json(out/'length_audit.json', result)
    print('Token lengths (raw training only):', result, flush=True)
    return selected


class TicketDataset:
    def __init__(self, rows, tokenizer, max_length, assistant_only):
        self.examples = [encode_training(r, tokenizer, assistant_only) for r in rows]
        if any(len(x['input_ids']) > max_length for x in self.examples):
            raise ValueError('Training sequence exceeds measured length; never silently truncate labels')

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


class Collator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, examples):
        import torch
        size = math.ceil(max(len(e['input_ids']) for e in examples)/8)*8
        batch = {}
        for key, padding in [('input_ids',self.tokenizer.pad_token_id), ('attention_mask',0), ('labels',-100)]:
            batch[key] = torch.tensor([e[key] + [padding]*(size-len(e[key])) for e in examples], dtype=torch.long)
        return batch


def predict(model, tokenizer, rows, path, rules=None, batch_size=8):
    """Exactly system/user -> native template -> greedy generate -> decode -> strip."""
    import torch
    from transformers import GenerationConfig
    if rules is not None and any(not r['id'].startswith('dv-') for r in rows):
        raise ValueError('Rules-in-prompt reference is restricted to dev')
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = 'left'
    generation = GenerationConfig(do_sample=False, num_beams=1, max_new_tokens=200,
        eos_token_id=model.generation_config.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    predictions = []
    with Path(path).open('w', encoding='utf-8') as file:
        for start in range(0,len(rows),batch_size):
            chunk = rows[start:start+batch_size]
            texts = []
            for row in chunk:
                messages = [{'role':m['role'],'content':m['content']} for m in row['messages'][:2]]
                if rules is not None:
                    messages[0]['content'] += '\n\n' + rules
                texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            inputs = tokenizer(texts, add_special_tokens=False, return_tensors='pt', padding=True).to('cuda')
            with torch.inference_mode():
                outputs = model.generate(**inputs, generation_config=generation)
            decoded = tokenizer.batch_decode(outputs[:,inputs['input_ids'].shape[1]:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
            for row, output in zip(chunk,decoded):
                record = {'id':row['id'], 'output':output.strip()}
                file.write(json.dumps(record,ensure_ascii=False)+'\n')
                predictions.append(record)
            file.flush()
            if (start//batch_size)%5 == 0:
                print(f'Generated {min(start+batch_size,len(rows))}/{len(rows)}: {Path(path).name}', flush=True)
    tokenizer.padding_side = 'right'
    return predictions


def measure(gold, predictions, path):
    from score import score
    result = score(gold, predictions)
    write_json(path,result)
    print('Scores', result['all'], flush=True)
    return result


def train_run(config, raw, clean, dev, tokenizer, max_length, revision, out):
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import Trainer, TrainingArguments, set_seed
    set_seed(SEED)
    torch.cuda.reset_peak_memory_stats()
    model = load_model(config.quantized, revision)
    base_parameters = sum(p.numel() for p in model.parameters())
    # Quantized tensors' physical shape does not equal their original parameter count.
    if not config.quantized:
        assert base_parameters <= 4_000_000_000
    if config.quantized:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.enable_input_require_grads()
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(r=config.rank, lora_alpha=2*config.rank, revision=revision,
        lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert trainable <= 50_000_000, trainable
    data = raw if config.data == 'raw' else clean
    dataset = TicketDataset(data, tokenizer, max_length, config.assistant_only)
    run_dir = out/'runs'/config.name
    run_dir.mkdir(parents=True,exist_ok=True)
    arguments = TrainingArguments(output_dir=str(run_dir/'checkpoints'),
        num_train_epochs=config.epochs, per_device_train_batch_size=2,
        gradient_accumulation_steps=8, learning_rate=config.lr, fp16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant':False},
        optim='adamw_torch', lr_scheduler_type='cosine', warmup_ratio=.05,
        weight_decay=.01, max_grad_norm=1.0, logging_steps=10,
        save_strategy='no', report_to='none', seed=SEED, data_seed=SEED,
        remove_unused_columns=False, dataloader_num_workers=0,
        group_by_length=True)
    trainer = Trainer(model=model,args=arguments,train_dataset=dataset,
        data_collator=Collator(tokenizer),processing_class=tokenizer)
    torch.cuda.synchronize()
    started = time.perf_counter()
    trainer.train()
    torch.cuda.synchronize()
    minutes = (time.perf_counter()-started)/60
    peak_allocated = torch.cuda.max_memory_allocated()/2**30
    peak_reserved = torch.cuda.max_memory_reserved()/2**30
    adapter = run_dir/'adapter'
    model.save_pretrained(adapter,safe_serialization=True)
    tokenizer.save_pretrained(adapter)
    write_json(run_dir/'training_log.json', trainer.state.log_history)
    # Drop optimizer and Trainer allocations before autoregressive inference.
    del trainer
    free_gpu()
    predictions = predict(model,tokenizer,dev,run_dir/'predictions_dev.jsonl')
    scores = measure(dev,predictions,run_dir/'scores.json')
    result = {'config':asdict(config), 'train_rows':len(data), 'trainable_parameters':trainable,
              'peak_gpu_gib':peak_allocated, 'peak_reserved_gib':peak_reserved,
              'training_minutes':minutes, 'scores':scores['all'], 'model_revision':revision}
    write_json(run_dir/'result.json',result)
    del model
    free_gpu()
    return result


def main(data_dir, out_dir, matrix=True, final_config=None):
    import torch
    from huggingface_hub import model_info
    from transformers import __version__ as transformers_version, set_seed
    from audit import audit
    started = time.perf_counter()
    if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
        raise RuntimeError('Assignment requires a free Colab NVIDIA T4 runtime')
    out, data = Path(out_dir), Path(data_dir)
    out.mkdir(parents=True,exist_ok=True)
    set_seed(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    audit(data/'train.jsonl',out)
    raw, clean, dev = load_rows(data/'train.jsonl'), load_rows(out/'train_clean.jsonl'), load_rows(data/'dev.jsonl')
    test = load_rows(data/'test_inputs.jsonl')
    assert len(dev)==200 and len(test)==400
    revision_file = out/'model_revision.txt'
    revision = revision_file.read_text().strip() if revision_file.exists() else model_info(MODEL_ID).sha
    revision_file.write_text(revision+'\n')
    tokenizer = load_tokenizer(revision)
    max_length = length_audit(raw,tokenizer,out)
    schema = Path('SCHEMA.md').read_text(encoding='utf-8')
    write_json(out/'environment.json',{'model_id':MODEL_ID,'revision':revision,'gpu':torch.cuda.get_device_name(0),
        'torch':torch.__version__,'transformers':transformers_version,'seed':SEED,
        'input_hashes':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [data/'train.jsonl',data/'dev.jsonl',data/'test_inputs.jsonl']}})
    # Both baselines use the same unquantized base weights; prompt is the only change.
    baseline_dir = out/'baselines'
    baseline_dir.mkdir(exist_ok=True)
    base = load_model(False,revision)
    for name, rules in [('fixed',None),('rules',schema)]:
        if (baseline_dir/f'{name}_scores.json').exists() and (baseline_dir/f'{name}_dev.jsonl').exists():
            saved_predictions = load_rows(baseline_dir/f'{name}_dev.jsonl')
            if [p['id'] for p in saved_predictions] == [r['id'] for r in dev]:
                print('Reusing completed baseline:', name, flush=True)
                continue
        predictions = predict(base,tokenizer,dev,baseline_dir/f'{name}_dev.jsonl',rules=rules)
        measure(dev,predictions,baseline_dir/f'{name}_scores.json')
    del base
    free_gpu()
    configs = RUNS if matrix else [Run(**(final_config or asdict(RUNS[0])))]
    results = [train_run(c,raw,clean,dev,tokenizer,max_length,revision,out) for c in configs]
    # Selection uses clean dev only, and never the hidden test.
    best = max(results,key=lambda r:(r['scores']['exact_match'],r['scores']['mean_field_acc'], -r['peak_gpu_gib']))
    write_json(out/'results.json',results)
    write_json(out/'best_config.json',best['config'])
    source = out/'runs'/best['config']['name']/'adapter'
    final = out/'adapter'
    shutil.copytree(source,final,dirs_exist_ok=True)
    shutil.copyfile(out/'runs'/best['config']['name']/'predictions_dev.jsonl',out/'predictions_dev.jsonl')
    from peft import PeftModel
    model = PeftModel.from_pretrained(load_model(best['config']['quantized'],revision),final)
    predictions = predict(model,tokenizer,test,out/'predictions_test.jsonl')
    assert len(predictions)==len(test) and [p['id'] for p in predictions]==[r['id'] for r in test]
    # Reloaded final adapter must regenerate a sample byte-for-byte under greedy decoding.
    # Batch size 8 here matches prediction production; evaluators may use different batch sizes.
    sample = predict(model,tokenizer,test[:8],out/'repro_sample.jsonl')
    assert sample == predictions[:8], 'Saved adapter predictions did not reproduce'
    del model
    free_gpu()
    write_json(out/'runtime.json',{'total_minutes':(time.perf_counter()-started)/60,'matrix':matrix})
    from reporting import build_report
    build_report(data,out)
    print('Final adapter and 400 raw test predictions saved:',out,flush=True)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data',default='data')
    parser.add_argument('--out',default='artifacts')
    parser.add_argument('--final-only',action='store_true')
    parser.add_argument('--config')
    args = parser.parse_args()
    config = json.loads(Path(args.config).read_text()) if args.config else None
    main(args.data,args.out,matrix=not args.final_only,final_config=config)


In [ ]:
%%writefile reporting.py
"""Build a concise report from measured outputs, never from invented metrics."""
import json
from pathlib import Path
import shutil
from pipeline import MODEL_ID, COMPARISONS, load_rows, write_json
from score import KEYS, parse, same, score


def failure_reason(fields, ticket, pred):
    if pred is None:
        return 'Output was not a JSON object; strict scorer marks every field wrong.'
    explanations = []
    for field in fields:
        explanations.append({
            'category':'Intent classification failed; inspect fraud precedence and overlapping payment/refund language.',
            'priority':'The model missed the ordered fraud, legal-threat or amount-threshold rule.',
            'amount_paise':'Amount extraction or integer rupee-to-paise conversion failed; check distracting balances and number formats.',
            'txn_date':'Transaction date extraction or calendar arithmetic failed; check Received, omitted years and quoted dates.',
            'txn_id':'NimbusPay ID extraction or normalization failed; check bank/order references.',
            'channel':'The model inferred a mode incorrectly or missed an explicit payment mode.',
            'language':'English/Hinglish classification failed, possibly due to short or mixed wording.',
            'needs_human':'The model missed P1 escalation or prior support contact.'
        }[field])
    return ' '.join(dict.fromkeys(explanations))


def build_report(data,out):
    out = Path(out)
    audit = json.loads((out/'audit_summary.json').read_text())
    results = json.loads((out/'results.json').read_text())
    best = json.loads((out/'best_config.json').read_text())
    dev = load_rows(Path(data)/'dev.jsonl')
    predictions = load_rows(out/'predictions_dev.jsonl')
    scores = score(dev,predictions)
    write_json(out/'final_dev_scores.json',scores)
    lookup = {r['id']:r['output'] for r in predictions}
    failures = []
    for row in dev:
        gold = json.loads(row['messages'][-1]['content'])
        prediction = parse(lookup.get(row['id']))
        wrong = [k for k in KEYS if prediction is None or k not in prediction or not same(prediction[k],gold[k])]
        if wrong or prediction is not None and set(prediction)!=set(KEYS):
            failures.append({'id':row['id'],'ticket':row['messages'][1]['content'],'gold':gold,
                'raw_output':lookup.get(row['id']), 'wrong_fields':wrong,
                'explanation':failure_reason(wrong,row['messages'][1]['content'],prediction)})
    write_json(out/'dev_failures.json',failures)
    # Favor diverse failed fields, then include further failures in dataset order.
    selected, covered = [], set()
    for failure in failures:
        if set(failure['wrong_fields'])-covered:
            selected.append(failure)
            covered.update(failure['wrong_fields'])
    selected = (selected+[f for f in failures if f not in selected])[:10]
    lengths = json.loads((out/'length_audit.json').read_text())
    best_result = next(r for r in results if r['config']==best)
    lines = ['# NimbusPay ticket triage — measured results', '',
        f'Base: `{MODEL_ID}` (494M parameters). Best: `{best["name"]}`; 4-bit load: **{best["quantized"]}**. '
        f'Dev exact match **{scores["all"]["exact_match"]:.1%}**, mean field accuracy **{scores["all"]["mean_field_acc"]:.1%}**.', '',
        '## Audit and choices', '',
        f'Only train.jsonl and SCHEMA.md informed cleaning. {audit["input_rows"]} rows → {audit["retained_rows"]}; '
        f'{audit["dropped_rows"]} dropped, {audit["fixed_rows"]} fixed. Findings: '+
        ', '.join(f'{k}={v}' for k,v in audit['findings'].items())+'. Counts of repairs overlap.',
        'Exact system/user duplicates retain the first occurrence. Invalid/truncated JSON and placeholder inputs are dropped rather than reconstructed. '
        'Legacy priority/date fields and IDs are normalized. Explicit claim amounts use Decimal arithmetic; balances and limits are excluded. '
        'Priority is recalculated from the ordered thresholds and legal threats. Categories/language are retained rather than relabelled with an AI model. '
        'Quoted threads are kept in training inputs to teach distraction handling. cleaning_log.csv records every changed row; audit.py provides executable evidence.', '',
        '## Training and controlled comparisons', '',
        'Hugging Face Transformers + PEFT + bitsandbytes keeps the training loop inspectable and saves standard PEFT adapters. '
        f'Raw training token lengths: median={lengths["p50"]}, p95={lengths["p95"]}, max={lengths["max"]}; '
        f'length={lengths["sequence_length"]}, with no training truncation. The native Qwen chat template supports system messages. '
        'Prompt and padding tokens are masked; assistant JSON and end-of-turn tokens receive loss. No packing. '
        'All runs use seed 2026, batch 2 × accumulation 8, two epochs, AdamW, LR 2e-4, cosine schedule, 5% warmup, '
        'dropout .05, and projection-module LoRA (alpha/r=2). Two epochs balance limited data and time; '
        f'best trainable parameters={best_result["trainable_parameters"]:,}. Dev is evaluation-only. '
        'Selecting the winner on dev can overfit dev; hidden test is never used for selection.', '',
        '| Run | Change relative to clean_qlora | Dev exact | Mean fields | Peak GPU GiB | Train min |',
        '|---|---|---:|---:|---:|---:|']
    changes = {'clean_qlora':'reference','raw_qlora':'raw data only','clean_lora':'16-bit weights only',
               'rank8':'rank 8 only; alpha/r fixed','full_loss':'all-token loss only'}
    for r in results:
        lines.append(f'| {r["config"]["name"]} | {changes.get(r["config"]["name"],"custom")} | '
            f'{r["scores"]["exact_match"]:.1%} | {r["scores"]["mean_field_acc"]:.1%} | '
            f'{r["peak_gpu_gib"]:.2f} | {r["training_minutes"]:.2f} |')
    if len(results)<5:
        lines.append('\nOnly the listed runs were executed; the four-comparison requirement remains incomplete until the full matrix is run.')
    lines += ['', 'Both base-model baselines are measured with score.py:', '', '| Prompt | Dev exact | Mean fields |', '|---|---:|---:|']
    for name in ['fixed','rules']:
        s = json.loads((out/'baselines'/f'{name}_scores.json').read_text())['all']
        lines.append(f'| {name} | {s["exact_match"]:.1%} | {s["mean_field_acc"]:.1%} |')
    weakest = sorted(scores['all']['fields'].items(),key=lambda x:x[1])[:3]
    categories = sorted(((k,v) for k,v in scores.items() if k.startswith('category=')),key=lambda kv:kv[1]['exact_match'])[:3]
    lines += ['', '## Error analysis', '',
        'Weakest fields: '+', '.join(f'{k} {v:.1%}' for k,v in weakest)+'. Weakest categories: '+
        ', '.join(f'{k.split("=")[1]} {v["exact_match"]:.1%} (n={v["n"]})' for k,v in categories)+'. '
        'Explanations below identify the failed rule; they are diagnostic hypotheses, not causal proof. Full tickets, labels and raw outputs are in dev_failures.json.', '',
        '| Dev ID | Specific mismatch (gold → prediction) | Explanation |', '|---|---|---|']
    for f in selected:
        p = parse(f['raw_output'])
        mismatch = '; '.join(f'{k}: {json.dumps(f["gold"][k])} → {json.dumps(p.get(k) if p else None)}' for k in f['wrong_fields'])
        if not mismatch:
            mismatch = 'Extra JSON keys'
        # Keep the report compact while the accompanying JSON preserves every field.
        explanation = f['explanation'].split('. ')[0]+'.'
        lines.append(f'| {f["id"]} | {mismatch} | {explanation} |')
    if len(selected)<10:
        lines.append(f'\nThere were only {len(failures)} observed failures; no additional failures have been invented.')
    lines += ['', '## Next week, reproduction and AI assistance', '',
        'Next: inspect the 10 failures, construct train-only boundary/format examples from SCHEMA.md, '
        'test a 1.5B model, and run three seeds with confidence intervals. Hold out a training-only validation split before further dev selection. '
        'Check calendar arithmetic and adversarial multi-transaction wording separately. More rank alone is unlikely to solve systematic extraction errors.',
        'Inference is native chat formatting followed by plain greedy model.generate(max_new_tokens=200). '
        'Decode excludes control tokens; whitespace stripping is the only output transformation. No rules/examples, constrained decoding, field filling, '
        'external model or API is used to produce labels/predictions. The final saved adapter is reloaded before test generation and its first eight outputs are checked for reproducibility.',
        'OpenAI Codex assisted with audit code, training/notebook implementation, local verification, Colab operation and this report generator. '
        'All scores, memory and timing figures come from the executed pipeline. Training-label repairs use deterministic code and the provided schema; '
        'Codex was not used as a labelling or prediction model. The candidate must review and be able to explain each line.']
    (out/'report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
    (out/'README.md').write_text(f'# NimbusPay ticket triage\n\nBase model: `{MODEL_ID}`\n\n'
        f'Pinned revision: `{best_result["model_revision"]}`\n\nFinal base loaded in 4-bit: **{best["quantized"]}**\n\n'
        f'Best run `{best["name"]}`: dev exact match **{scores["all"]["exact_match"]:.1%}**, '
        f'mean field accuracy **{scores["all"]["mean_field_acc"]:.1%}**.\n\n'
        'Open notebook.ipynb in a free Colab T4 runtime. Upload the original candidate_data.zip when requested; '
        'the training data and base weights are intentionally excluded from Git. Read report.md for methods and measured experiments. '
        'Source modules are embedded in the notebook and also provided as .py files for inspection. '
        'Keep the repository private until the verified deadline and stop committing before that deadline.\n',encoding='utf-8')
    shutil.copyfile(out/'cleaning_log.csv',Path('cleaning_log.csv'))


In [ ]:
%%writefile score.py
"""Score triage predictions against labelled data.

    python score.py --gold data/dev.jsonl --pred my_dev_predictions.jsonl

Prediction file: one JSON object per line
    {"id": "dv-00001", "output": "<the raw text your model generated>"}

Scoring is strict:
  * "output" must be ONLY a JSON object (leading/trailing whitespace is ignored).
  * Every field must match exactly, including type (true is not 1, 129900 is not "129900").
  * A missing id or invalid JSON counts as wrong on every field.
"""
import argparse, json
from collections import defaultdict

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]


def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def same(a, b):
    return type(a) is type(b) and a == b


def parse(text):
    try:
        obj = json.loads(text.strip())
    except (json.JSONDecodeError, AttributeError):
        return None
    return obj if isinstance(obj, dict) else None


def score(gold_rows, pred_rows):
    preds = {p["id"]: p.get("output") for p in pred_rows}
    groups = defaultdict(lambda: {"n": 0, "valid": 0, "exact": 0, "field": defaultdict(int)})
    for g in gold_rows:
        gold = json.loads(g["messages"][-1]["content"])
        pred = parse(preds.get(g["id"]))
        names = ["all"] + ([f"slice={g['slice']}"] if "slice" in g else []) + [f"category={gold['category']}"]
        hits = {k: pred is not None and k in pred and same(pred[k], gold[k]) for k in KEYS}
        exact = pred is not None and all(hits.values()) and set(pred) == set(KEYS)
        for name in names:
            s = groups[name]
            s["n"] += 1
            s["valid"] += pred is not None
            s["exact"] += exact
            for k in KEYS:
                s["field"][k] += hits[k]
    out = {}
    for name, s in groups.items():
        n = s["n"]
        fields = {k: s["field"][k] / n for k in KEYS}
        out[name] = {"n": n, "json_valid": s["valid"] / n, "exact_match": s["exact"] / n,
                     "mean_field_acc": sum(fields.values()) / len(KEYS), "fields": fields}
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gold", required=True)
    ap.add_argument("--pred", required=True)
    ap.add_argument("--json", action="store_true", help="print machine-readable JSON")
    a = ap.parse_args()
    res = score(load(a.gold), load(a.pred))
    if a.json:
        print(json.dumps(res, indent=1))
        return
    print(f"{'group':28} {'n':>5} {'valid':>7} {'exact':>7} {'fields':>7}")
    for name in sorted(res, key=lambda x: (x != "all", x)):
        r = res[name]
        print(f"{name:28} {r['n']:5d} {r['json_valid']:7.1%} {r['exact_match']:7.1%} {r['mean_field_acc']:7.1%}")
    print("\nper-field accuracy (all rows)")
    for k, v in res["all"]["fields"].items():
        print(f"  {k:14} {v:6.1%}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile SCHEMA.md
# NimbusPay Triage Labelling Spec (v2, current)

NimbusPay is a fictional payments app. Every incoming support ticket is turned into one JSON object, the *triage record*. This document is the single source of truth for what a correct record looks like. **All dev and test labels follow this spec exactly.**

## Input

```
Received: 2026-08-14 (Fri)
Source: email

<the customer's message>
```

`Received` is the date the ticket arrived. `Source` is one of `app-chat`, `email`, `twitter`, `ivr-transcript`.

## Output

A single JSON object with **exactly these 8 keys**. Use `null` when a value is not present in the ticket.

```json
{"category": "refund", "priority": "P2", "amount_paise": 2936300, "txn_date": "2026-09-13", "txn_id": "NP5972758064", "channel": "netbanking", "language": "en", "needs_human": false}
```

| Key | Type | Allowed values |
|---|---|---|
| `category` | string | `payment_failed`, `refund`, `fraud`, `account_access`, `kyc`, `offers`, `other` |
| `priority` | string | `P1`, `P2`, `P3` |
| `amount_paise` | integer or null | amount in paise (₹1 = 100 paise) |
| `txn_date` | string or null | `YYYY-MM-DD` |
| `txn_id` | string or null | `NP` followed by 10 digits |
| `channel` | string or null | `upi`, `card`, `netbanking`, `wallet` |
| `language` | string | `en`, `hi-en` |
| `needs_human` | boolean | `true`, `false` |

## Rules

### category
| Value | Use when |
|---|---|
| `fraud` | The customer says a transaction (payment, cash withdrawal, direct debit) happened that **they did not make, approve or recognise**. This wins over every other category. |
| `payment_failed` | A payment, transfer or top-up the customer made failed, was declined, is stuck, or did not reach the recipient. |
| `refund` | Money is owed back: asking for a refund, a cancelled order, a return, the same transaction charged more than once, a refund that has not arrived. |
| `account_access` | Cannot get in: locked or blocked account, blocked PIN, forgotten passcode or password, OTP not received, reset problems. |
| `kyc` | Identity verification: how to verify, why it is needed, verification that failed or is pending, documents, Aadhaar/PAN, video KYC. |
| `offers` | Cashback, promo codes, coupons, scratch cards, referral bonuses, reward points. |
| `other` | Anything else: card delivery, exchange rates, closing the account, general how-to questions. |

### priority (apply in this order)
1. **`P1`** if any of these is true:
   - `category` is `fraud`
   - the disputed amount is **₹50,000 or more**
   - the customer makes a **legal threat**: mentions RBI, an ombudsman, consumer court, a lawyer, a legal notice or legal action
2. **`P2`** if not P1 and either:
   - `category` is `payment_failed` or `refund` **and** the amount is **₹5,000 or more**
   - `category` is `account_access`
3. **`P3`** otherwise.

### Transaction fields
`amount_paise`, `txn_date`, `txn_id` and `channel` describe one specific transaction. For `account_access`, `kyc` and `other` tickets they are always `null`.

### amount_paise
- The amount of the transaction or claim the ticket is **about**, converted to paise as an integer.
- Ignore other amounts: account balance, limits, fees.
- `k` means thousand (`1.3k` = ₹1,300). `lakh` means 100,000 (`1.5 lakh` = ₹1,50,000).
- `null` if no such amount is stated.

### txn_date
- The date of the transaction the ticket is about, as `YYYY-MM-DD`.
- Resolve relative dates ("yesterday", "3 days ago", "kal", "parso") against the `Received` date.
- If the year is missing, use the most recent such date on or before the `Received` date.
- Dates inside quoted earlier emails (lines starting with `>`) are not the transaction date.
- `null` if the customer does not say when it happened.

### txn_id
- NimbusPay transaction ids are `NP` + 10 digits. Normalize to uppercase with no spaces, hyphens or symbols: `np 12345 67890` → `NP1234567890`.
- Other reference numbers (ticket numbers, bank reference numbers, order numbers) are **not** transaction ids.
- `null` if no NimbusPay transaction id is given.

### channel
- The payment mode of the transaction, **only when it is explicitly stated**: `upi`, `card` (credit card, debit card, "card payment", Visa, Mastercard), `netbanking` (also "internet banking"), `wallet`.
- `null` otherwise.

### language
- `hi-en` if the customer's message is written in Hinglish (Hindi in Latin script, possibly mixed with English words).
- `en` otherwise.

### needs_human
`true` if `priority` is `P1`, **or** the customer says they have already contacted support about this issue before (for example "third time", "again and again", "baar baar"). Otherwise `false`.


In [ ]:
%%writefile requirements.txt
transformers==4.51.3
peft==0.15.2
accelerate==1.6.0
bitsandbytes==0.48.2
safetensors==0.5.3
huggingface-hub==0.30.2


## Audit

Deterministic train-only repairs and a per-row cleaning log. Categories and language are retained. Malformed labels and junk tickets are dropped. Long quoted threads remain to teach distraction handling.

In [ ]:
from audit import audit
import json
summary = audit(WORK/"data/train.jsonl", WORK/"artifacts")
print(json.dumps(summary,indent=2))

## Baselines, training and experiment matrix

Five independent runs form four comparisons against clean QLoRA: raw vs cleaned data; 16-bit LoRA vs 4-bit QLoRA; rank 8 vs 16 (alpha/r=2); and all-token vs assistant-only loss. Each starts from fresh base weights and seed 2026. No dev/test rows receive training loss. Elapsed training time and peak allocated GPU memory are measured separately from inference. Use final-only reproduction after the best configuration is known; full experiment code stays included.

In [ ]:
from pipeline import main
RUN_EXPERIMENTS = True
BEST_CONFIG = None  # Filled with the measured winner after completing the matrix
main(WORK/"data", WORK/"artifacts", matrix=RUN_EXPERIMENTS, final_config=BEST_CONFIG)

## Measured results and error analysis

All figures below are read from executed model outputs. The report selects ten observed dev failures with diverse fields; full details are preserved in dev_failures.json.

In [ ]:
from IPython.display import display, Markdown
display(Markdown((WORK/"artifacts/report.md").read_text()))
print("Wall-clock minutes:", (WORK/"artifacts/runtime.json").read_text())

## Save the submission

Download the submission archive and this notebook with its outputs. The archive excludes all training data and base model weights. Save adapters/logs promptly if Colab warns of disconnection. Keep GitHub private until the verified deadline and do not commit after it.

In [ ]:
import shutil
SUBMIT = WORK/"submission"
SUBMIT.mkdir(exist_ok=True)
for name in ["audit.py", "pipeline.py", "reporting.py", "score.py", "SCHEMA.md", "requirements.txt"]:
    shutil.copyfile(WORK/name,SUBMIT/name)
for name in ["README.md", "report.md", "cleaning_log.csv", "predictions_test.jsonl", "predictions_dev.jsonl", "dev_failures.json", "audit_summary.json", "length_audit.json", "results.json", "best_config.json", "environment.json", "model_revision.txt", "runtime.json"]:
    shutil.copyfile(WORK/"artifacts"/name,SUBMIT/name)
shutil.copytree(WORK/"artifacts/adapter", SUBMIT/"adapter",dirs_exist_ok=True)
for folder in ["runs", "baselines"]:
    shutil.copytree(WORK/"artifacts"/folder,SUBMIT/"artifacts"/folder,dirs_exist_ok=True)
(SUBMIT/".gitignore").write_text("data/\n*.zip\n__pycache__/\nbase_model/\n**/checkpoints/\n")
assert (SUBMIT/"adapter/adapter_config.json").exists()
assert (SUBMIT/"adapter/adapter_model.safetensors").exists()
assert len((SUBMIT/"predictions_test.jsonl").read_text().splitlines()) == 400
archive = shutil.make_archive(str(WORK/"nimbuspay_submission"), "zip", SUBMIT)
files.download(archive)
print("Also choose File > Download > Download .ipynb to preserve notebook outputs.")